# Keep your application and swap the harness

Use the same model, Python tool, MCP server, and conversation flow across installed harnesses. Optionally enable Temporal.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/10_harness_switch.ipynb)

Run these cells in order in **Google Colab**. Everything runs in its cloud runtime:
no repository checkout or laptop installation. A CPU runtime is enough.
Model calls use your provider account. Clear outputs before sharing a saved copy.

## Install and choose a model

Keep the defaults for a first run. To switch providers, change `MODEL`:
- OpenAI: `openai/gpt-5.4-mini` with `OPENAI_API_KEY`.
- Anthropic: `anthropic/claude-sonnet-4-6` with `ANTHROPIC_API_KEY`.
- OpenRouter: `openrouter/anthropic/claude-sonnet-4.6` with `OPENROUTER_API_KEY`.

LiteLLM's Python SDK handles provider translation; no gateway is required.
Optional: set `API_BASE` to your gateway URL and use its exact model alias
with `LITELLM_API_KEY`. Leave `API_BASE` empty for direct provider access.

For Gemini, Groq, Mistral, DeepSeek, Together AI, xAI, Azure, Bedrock,
Vertex AI, or Ollama, see the [model setup guide](https://github.com/BerriAI/liteagents/blob/main/docs/models.md).
Cloud authentication and provider-specific environment variables must be configured
in this runtime before running the agent.

In [ ]:
import os

HARNESS = os.environ.get("LITEAGENTS_HARNESS", "deepagents")
MODEL = os.environ.get("LITEAGENTS_MODEL", "openai/gpt-5.4-mini")
API_BASE = os.environ.get("LITEAGENTS_API_BASE", "")  # Optional gateway URL.

HARNESSES = [HARNESS]  # Try ["deepagents", "pydantic-ai"] or any of the six below.

Available harnesses: `deepagents`, `pydantic-ai`, `claude-sdk`, `codex`,
`opencode-v1`, `opencode-v2`. Rerun the install cell after changing your selection.
Packages are reused within this runtime; a fresh Colab runtime needs its own install.
OpenCode is installed only when selected.

This preview installs from a GitHub release wheel because the PyPI name currently
belongs to another package. It does not clone the repository.

In [ ]:
# @title Install selected integrations
import shutil
import subprocess
import sys

selected_harnesses = HARNESSES
extras = sorted(set(selected_harnesses) | {"mcp", "temporal"})
release = "https://github.com/BerriAI/liteagents/releases/download/v0.3.0a3"
package = f"liteagents[{','.join(extras)}] @ {release}/liteagents-0.3.0a3-py3-none-any.whl"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", package,
    "-c", f"{release}/constraints-tested.txt",
])
if any(h.startswith("opencode-") for h in selected_harnesses):
    if shutil.which("opencode") is None:
        subprocess.check_call(["npm", "install", "-g", "opencode-ai@1.18.29"])
    subprocess.check_call(["opencode", "--version"])
print("Ready:", ", ".join(selected_harnesses))

## Add your API key

In Colab, open the **key icon → Secrets**, add the key named above, and enable
notebook access. Or enter it in the hidden prompt below. The key stays out of your
code and saved outputs. If installation asks for a runtime restart,
restart once and run the cells again.

In [ ]:
# @title Connect your provider
from getpass import getpass

KEY_NAME = "LITELLM_API_KEY" if API_BASE else {
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "together_ai": "TOGETHERAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "xai": "XAI_API_KEY",
    "azure": "AZURE_API_KEY",
}.get(MODEL.split("/", 1)[0])
API_KEY = None
if KEY_NAME:
    API_KEY = os.environ.get(KEY_NAME)
    if not API_KEY:
        try:
            from google.colab import userdata
        except ImportError:
            pass
        else:
            try:
                API_KEY = userdata.get(KEY_NAME)
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
    API_KEY = API_KEY or getpass(f"{KEY_NAME}: ")
    if not API_KEY:
        raise ValueError(f"Provide {KEY_NAME} before running the agent.")
    os.environ[KEY_NAME] = API_KEY
else:
    print("Using provider credentials from the runtime; see the model setup guide.")
os.environ["LITEAGENTS_MODEL"] = MODEL
if API_BASE:
    os.environ["LITEAGENTS_API_BASE"] = API_BASE
MODEL_KWARGS = {"api_base": API_BASE, "api_key": API_KEY} if API_BASE else {}

## Create your profile

This is the SDK interface. The following cells change this profile to demonstrate
one feature. A temporary workspace keeps each run's files separate.

In [ ]:
import asyncio
import tempfile
from pathlib import Path
from uuid import uuid4

from liteagents import LiteAgentClient, LiteAgentOptions, ProfileOptions

profile = ProfileOptions(
    harness=HARNESS,
    model=MODEL,
    model_kwargs=MODEL_KWARGS,
    tools=[],
    max_turns=10,
    system_prompt="Use the requested tools and report their actual results. Be concise.",
)
workspace_root = Path(os.environ.get(
    "LITEAGENTS_NOTEBOOK_WORKSPACE", Path(tempfile.gettempdir()) / "liteagents-notebooks"
))
workspace_root.mkdir(parents=True, exist_ok=True)
workspace = Path(tempfile.mkdtemp(prefix="run-", dir=workspace_root)).resolve()
print("Workspace:", workspace)

In [ ]:
USE_TEMPORAL = False  # True starts a demo Temporal service in this runtime.

The application below uses the same model, Python tool, MCP server, and follow-up
for each harness in `HARNESSES`. Set that list above and rerun the install cell.
Optional Temporal starts here with an in-process worker. For separate worker crash
recovery, try the durable notebook.

## Start a demo Temporal service

This cell downloads and starts Temporal **inside the Colab runtime**.
No terminal or separate account is needed. A runtime reset deletes local checkpoints;
persistent deployments need an external [Temporal service](https://github.com/BerriAI/liteagents/blob/main/docs/self-hosting.md).
Set `LITEAGENTS_TEMPORAL_ADDRESS` to use an existing development service instead.

In [ ]:
temporal_environment = globals().get("temporal_environment")
temporal_address = os.environ.get("LITEAGENTS_TEMPORAL_ADDRESS")
if USE_TEMPORAL:
    from temporalio.testing import WorkflowEnvironment

    if temporal_environment is not None:
        temporal_address = temporal_environment.client.service_client.config.target_host
    elif not temporal_address:
        temporal_environment = await WorkflowEnvironment.start_local(
            ui=False,
            dev_server_existing_path=shutil.which("temporal"),
            dev_server_database_filename=str(workspace / "temporal.sqlite"),
        )
        temporal_address = temporal_environment.client.service_client.config.target_host
    print("Temporal:", temporal_address)

In [ ]:
mcp_server = workspace / "mcp_server.py"
mcp_server.write_text(r'''
"""Local MCP demo server: no account or external service needed."""

try:
    from mcp.server.mcpserver import MCPServer
except ImportError:  # MCP 1.x
    from mcp.server.fastmcp import FastMCP as MCPServer

server = MCPServer("Orders")


@server.tool()
def lookup_order(order_id: str) -> dict:
    """Look up a demo order's payment status and total."""
    return {"order_id": order_id, "total_usd": 12, "status": "paid"}


if __name__ == "__main__":
    server.run()
''')
print("Demo MCP server:", mcp_server)

In [ ]:
from contextlib import AsyncExitStack
from typing import ClassVar

from liteagents import AssistantMessage, TemporalOptions, Tool, ToolUseBlock


class ShippingStatus(Tool):
    name = "shipping_status"
    description = "Return the shipping status of an order."
    input_schema: ClassVar[dict] = {
        "type": "object", "properties": {"order_id": {"type": "string"}},
        "required": ["order_id"], "additionalProperties": False,
    }

    def __init__(self):
        self.calls = 0

    async def execute(self, input):
        self.calls += 1
        return f"Order {input['order_id']} has shipped. Tracking code COBALT-42."

In [ ]:
profile.tools = ["shipping_status", "orders_lookup_order"]
profile.mcp_servers = {"orders": {
    "command": sys.executable,
    "args": [str(mcp_server)],
    "allowed_tools": ["lookup_order"],
}}
profile.temporal = (
    TemporalOptions(address=temporal_address, checkpoint_path=str(workspace / "checkpoints.sqlite"))
    if USE_TEMPORAL else None
)

## The application stays the same

The tool-name and follow-up assertions make it easy to spot differences while you experiment.
Only the harness selector changes in the comparison cell below.

In [ ]:
async def application(profile, cwd, *, durable):
    tool = ShippingStatus()
    async with AsyncExitStack() as stack:
        if durable:
            from liteagents.temporal import LiteAgentWorker

            # The demo owns a worker for convenience. In a deployment it runs
            # separately with this same profile and application tool registry.
            await stack.enter_async_context(
                LiteAgentWorker(profile=profile, cwd=cwd, tools=[tool]).running()
            )
        client = await stack.enter_async_context(LiteAgentClient(
            options=LiteAgentOptions(profile=profile, cwd=cwd, tools=[tool])
        ))
        run_id = uuid4().hex
        used = set()
        async for message in client.query(
            "Use orders_lookup_order and shipping_status for order A123. "
            "Report the payment total and tracking code from the actual results.", run_id=run_id,
        ):
            if isinstance(message, AssistantMessage):
                used.update(b.name for b in message.content if isinstance(b, ToolUseBlock))
        result = await (await client.get_run(run_id)).result()
        assert used == {"orders_lookup_order", "shipping_status"}, used
        assert "12" in result.text and "COBALT-42" in result.text, result.text
        calls = tool.calls
        followup_id = uuid4().hex
        async for _ in client.query(
            "From our conversation, repeat the payment total and tracking code. Do not call tools.",
            run_id=followup_id,
        ):
            pass
        answer = (await (await client.get_run(followup_id)).result()).text
        assert "12" in answer and "COBALT-42" in answer, answer
        assert tool.calls == calls, "Follow-up unexpectedly repeated the application tool"
        print(f"{profile.harness}: {answer}")
        return {"harness": profile.harness, "tools": sorted(used), "answer": answer}

In [ ]:
comparisons = []
for harness in HARNESSES:
    selected = profile.model_copy(update={"harness": harness}, deep=True)
    async with asyncio.timeout(180):
        comparisons.append(await application(selected, workspace, durable=USE_TEMPORAL))
comparisons

To explore native settings, set the selected harness's `harness_options` before running.
Those options are specific to that harness. Shared tools, model configuration, and MCP stay portable.
For file changes and independently checked test results, open the
[coding comparison notebook](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/compare_harnesses/compare.ipynb).

## Cleanup

Stop only this notebook's demo Temporal service, if one was started.

In [ ]:
if temporal_environment is not None:
    await temporal_environment.shutdown()
    temporal_environment = None